In [ ]:
import json
import time
from langdetect import detect
import requests
import xml.etree.ElementTree as ET
import pandas as pd
from google_play_scraper import app, exceptions
import base64
import json
#activity filename:AndroidManifest.xml in:file repo:liato/android-bankdroid

url = f"https://api.github.com/search/code?q=activity+filename:AndroidManifest.xml+repo:" # search only the file name by dropping activity

# Define a list of authorization headers
authorization_headers = [
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key2}"},
    {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key3}"}
]

def makeRequest(url):
    tries_flag = 0
    headers=   {'Accept': 'application/vnd.github+json',"Authorization": "Bearer {github_key1}"}
    while True:
        response = requests.get(url, headers=headers)
        if response.status_code == 200:
            return response.json()
        elif response.status_code == 403 and ('message' in response.json() and "rate limit" in response.json()["message"].lower()):
            tries_flag += 1
            if tries_flag > len(authorization_headers):
                reset_timestamp = int(response.headers['X-RateLimit-Reset'])
                current_timestamp = int(time.time())
                wait_time = reset_timestamp - current_timestamp
                print(f"Rate limit exceeded. Waiting for {wait_time} seconds...")
                time.sleep(wait_time + 1)
                tries_flag = 0
            else:
                # Cycle through the authorization headers
                headers = authorization_headers[tries_flag - 1]
        elif response.status_code == 422:
            return {"total_count": 0}
        elif response.status_code == 404:
            return {"total_count": 0}
        else:
            print(f"Request failed with status code {response.status_code}. Retrying... " + url)
            time.sleep(1)

df=pd.read_json('fdroid_apps_data.json')
df = df[df['owner'].notnull()]

for index, obj in df.iterrows():
    app_repo=obj.copy()
    response=makeRequest(url+str(obj["owner"])+"/"+str(obj["name"]))
    # Check if the file was found
    androidmanifest_count=response["total_count"]
    app_repo['androidmanifest_count']=androidmanifest_count
  
    packages=[]
    package_counts=0
    google_play_data=[]
    for item in response["items"]:
        # Get the URL of the  search result
        manifest_url = item["url"]
        manifest_path=item["path"]
        response =makeRequest(manifest_url)
        if 'content' in response:
            # Retrieve the content of the AndroidManifest.xml file
            manifest_xml = base64.b64decode(response['content']).decode('utf-8')
            package_value=""
            # Parse the XML file and retrieve the package value
            try:
                root = ET.fromstring(manifest_xml)
                package_value = root.attrib.get("package")
            except:
                google_play_data.append("") 
            if package_value is not None and package_value!="":
                package_counts+=1
                packages.append(package_value)
                try:
                    result = app(package_value.encode('utf-8'),lang=detect(package_value))
                    if result:
                        google_play_data.append(result)
                    else:
                        google_play_data.append("")
                except exceptions.NotFoundError:
                        google_play_data.append("")
        else:
            google_play_data.append("")
    app_repo['package_counts']=package_counts
    app_repo['packages']=packages
    app_repo['google_play_data']=google_play_data
    with open('../Data/fdroid_repos_packages.json', mode='a', encoding='utf-8') as json_file:
        json.dump(app_repo.to_dict(), json_file)
        json_file.write(',\n')
        print('done')